# MEDLEY-BENCH: Benchmark Any LLM

**Behavioral Metacognition Under Social Pressure**

This notebook runs the [MEDLEY-BENCH](https://pypi.org/project/medley-bench/) benchmark on any LLM.
It measures how models monitor, evaluate, and control their reasoning when challenged by peer disagreement.

**Three ways to benchmark a model:**

| Method | GPU needed? | Cost | Example |
|--------|-------------|------|---------|
| A. Cloud API (OpenRouter) | No | Free tier available | Llama 3.3 70B, Gemma 3 27B |
| B. Direct API (Anthropic/OpenAI/Google) | No | Pay-per-token | Claude, GPT, Gemini |
| C. Local model via vLLM (HuggingFace) | Yes (T4/A100) | Free (your GPU) | Any HF model |

Pick whichever section applies to you. Each is self-contained.

## Setup (required for all methods)

In [ ]:
!pip install -q medley-bench

In [ ]:
# Verify installation
!medley-bench about

---
## Method A: Cloud API via OpenRouter (easiest, free tier available)

OpenRouter provides access to 200+ models with a single API key.
Several models are available for **free** (rate-limited).

1. Get a free key at https://openrouter.ai/keys
2. Paste it below

In [ ]:
import os
import getpass

# Option 1: Enter key interactively
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# Option 2: If using Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
# Free models (no credit card needed)
# Change the model to any OpenRouter model ID
MODEL = "google/gemma-3-27b-it:free"  # free tier

# Other free options:
# MODEL = "meta-llama/llama-3.3-70b-instruct:free"
# MODEL = "mistralai/mistral-small-3.1-24b-instruct:free"

# Paid models (requires OpenRouter credit):
# MODEL = "anthropic/claude-haiku-4.5"       # ~$2-5 per full run
# MODEL = "openai/gpt-4.1"                   # ~$5-10 per full run
# MODEL = "google/gemini-2.5-flash"           # ~$1-3 per full run

In [ ]:
# Smoke test: run 3 instances to verify everything works
!medley-bench benchmark --models "{MODEL}" --n-instances 3 --output results_openrouter/

In [ ]:
# Full run (130 instances, ~30-90 min depending on model)
# Uncomment when ready:
# !medley-bench benchmark --models "{MODEL}" --output results_openrouter/

In [ ]:
!medley-bench leaderboard --results results_openrouter/

---
## Method B: Direct API (Anthropic, OpenAI, or Google)

Use your own API key for a specific provider. Lower latency than OpenRouter.

In [ ]:
import os
import getpass

# === Pick ONE provider and set its key ===

# Anthropic (Claude)
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
# MODEL = "claude-haiku-4.5"

# OpenAI (GPT)
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
# MODEL = "gpt-4.1-mini"

# Google (Gemini)
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")
MODEL = "gemini-2.5-flash"

print(f"Using: {MODEL}")

In [ ]:
# Smoke test
!medley-bench benchmark --models "{MODEL}" --n-instances 3 --output results_direct/

In [ ]:
# Full run (uncomment when ready)
# !medley-bench benchmark --models "{MODEL}" --output results_direct/

---
## Method C: Local HuggingFace Model via vLLM (GPU required)

Run any HuggingFace model locally using vLLM, which exposes an OpenAI-compatible API.
MEDLEY-BENCH connects to it via the Ollama provider with a custom base URL.

**Requirements:** Colab with GPU runtime (T4 for 7B models, A100 for 13B+).

To enable GPU: Runtime > Change runtime type > T4 GPU

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install vLLM (takes ~2 min)
!pip install -q vllm

In [ ]:
import os
import getpass

# HuggingFace token (needed for gated models like Llama, Gemma)
# Get yours at https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace token (or press Enter to skip): ") or ""

In [ ]:
# Choose a HuggingFace model
# Small models that fit on T4 (16GB VRAM):
HF_MODEL = "google/gemma-3-4b-it"   # 4B, ~8GB VRAM
# HF_MODEL = "Qwen/Qwen3-8B"          # 8B, ~16GB VRAM
# HF_MODEL = "microsoft/phi-4"         # 14B, ~16GB VRAM (quantized)

# Larger models (need A100 40GB+):
# HF_MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # 8B
# HF_MODEL = "google/gemma-3-27b-it"              # 27B, needs A100 80GB

print(f"Model: {HF_MODEL}")

In [ ]:
# Start vLLM server in the background
# It exposes an OpenAI-compatible API on port 8000
import subprocess
import time

vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", HF_MODEL,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", "8192",
        "--dtype", "auto",
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

print(f"Starting vLLM server (pid={vllm_proc.pid})...")
print("This may take 2-5 minutes to download and load the model.")

# Wait for server to be ready
import urllib.request
for i in range(120):  # wait up to 10 min
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print(f"vLLM server ready after {i*5}s!")
        break
    except Exception:
        time.sleep(5)
        if i % 6 == 0:
            print(f"  waiting... ({i*5}s)")
else:
    print("Timeout! Check /tmp/vllm.log for errors:")
    !tail -20 /tmp/vllm.log

In [ ]:
# Quick test: verify the vLLM server responds
!curl -s http://localhost:8000/v1/models | python -m json.tool

In [ ]:
# Run MEDLEY-BENCH against the local vLLM server.
# We use the Ollama provider with a custom base URL pointing at vLLM.
import os
os.environ["OLLAMA_BASE_URL"] = "http://localhost:8000"

# The model name must match what vLLM reports (usually the HF model ID)
BENCH_MODEL = f"ollama/{HF_MODEL}"
print(f"Benchmarking: {BENCH_MODEL}")

In [ ]:
# Smoke test: 3 instances
!medley-bench benchmark --models "{BENCH_MODEL}" --n-instances 3 --output results_vllm/

In [ ]:
# Full run (uncomment when ready)
# !medley-bench benchmark --models "{BENCH_MODEL}" --output results_vllm/

In [ ]:
!medley-bench leaderboard --results results_vllm/

In [ ]:
# Stop vLLM server when done
vllm_proc.terminate()
print("vLLM server stopped.")

---
## Adding a Live Judge (optional, recommended)

By default, MEDLEY-BENCH scores only the **deterministic measures** (T1 + most of T2 = 75% of total).
To score the **judge-dependent measures** (T3 = 25%), add a judge model.

**Recommended judge:** Gemini 2.5 Flash (fast, cheap, excellent structured output).

In [ ]:
import os
import getpass

# Set Google API key for the judge
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key for judge: ")

In [ ]:
# Re-run with live judge (use whichever MODEL you set above)
# Replace MODEL with your chosen model from Method A, B, or C

!medley-bench benchmark \
  --models "{MODEL}" \
  --judge-model gemini-2.5-flash \
  --judge-api-key $GOOGLE_API_KEY \
  --n-instances 3 \
  --output results_with_judge/

In [ ]:
!medley-bench leaderboard --results results_with_judge/

---
## Inspect Results

Results are saved as JSON files. Each file contains per-instance scores,
computed measures, and raw model responses.

In [ ]:
import json
import glob

# Find all result files
result_files = glob.glob("results_*/*.json")
print(f"Found {len(result_files)} result file(s):")

for f in result_files:
    data = json.load(open(f))
    valid = [r for r in data if "error" not in r and "total_score" in r]
    if not valid:
        print(f"  {f}: {len(data)} instances (no valid scores)")
        continue
    avg = sum(r["total_score"] for r in valid) / len(valid)
    t1 = sum(r["tier_scores"]["reflective_updating"] for r in valid) / len(valid)
    t2 = sum(r["tier_scores"]["social_robustness"] for r in valid) / len(valid)
    t3 = sum(r["tier_scores"]["epistemic_articulation"] for r in valid) / len(valid)
    model = valid[0].get("model", "unknown")
    print(f"  {model}: total={avg:.3f}  T1={t1:.3f}  T2={t2:.3f}  T3={t3:.3f}  (n={len(valid)})")

In [ ]:
# Look at a single instance in detail
if result_files:
    data = json.load(open(result_files[0]))
    r = next(r for r in data if "error" not in r)
    print(f"Instance: {r['instance_id']} ({r.get('domain')})")
    print(f"Total: {r['total_score']:.4f}")
    print(f"\nTier scores:")
    for k, v in r["tier_scores"].items():
        print(f"  {k}: {v:.4f}")
    print(f"\nKey computed measures:")
    for k in ["proportionality", "private_vs_social_delta", "epistemic_cowardice", "content_engagement"]:
        v = r.get("computed", {}).get(k)
        if v is not None:
            print(f"  {k}: {v:.4f}")
    print(f"\nStep A response (first 500 chars):")
    print(r["raw_responses"]["step_a"][:500])

---

**Links:**
- PyPI: https://pypi.org/project/medley-bench/
- GitHub: https://github.com/ki-smile/medley-bench
- Dataset: https://www.kaggle.com/datasets/farhadabtahi/medley-bench-data
- Preprint: Abtahi et al. (2026) "MEDLEY-BENCH: Scale Buys Evaluation but Not Control in AI Metacognition"

**More help:** Run `!medley-bench examples` for additional usage patterns.